# Visualisation du Dataset MedQA-MA
Distribution des spécialités, longueur des textes, équilibre train/val/test.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

BASE = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
CSV_PATH = os.path.join(BASE, 'data', 'raw',
    'MedQA-MA; Question Answering Dataset in Moroccan A',
    'Dataset', 'MedQA_Ma dataset', 'MedQA_MA.csv')

df = pd.read_csv(CSV_PATH)
df['Category'] = df['Category'].str.strip()
df['q_len'] = df['Question'].str.split().str.len()
df['a_len'] = df['Answer'].str.split().str.len()

print(f"Lignes : {len(df):,}  |  Spécialités : {df['Category'].nunique()}")
df.head(3)

## 1. Distribution des spécialités médicales

In [ ]:
counts = df['Category'].value_counts()

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(counts.index, counts.values,
               color=sns.color_palette('Blues_r', len(counts)))
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_xlabel('Nombre de questions')
ax.set_title('Distribution des spécialités médicales — MedQA-MA', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'nlp', 'preprocessing', 'results', 'specialty_distribution.png'), dpi=150)
plt.show()

## 2. Équilibre des classes (Top 10)

In [ ]:
top10 = counts.head(10)

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    top10.values,
    labels=top10.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('Set2', len(top10))
)
for t in autotexts:
    t.set_fontsize(8)
ax.set_title('Top 10 spécialités (proportion)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'nlp', 'preprocessing', 'results', 'top10_pie.png'), dpi=150)
plt.show()

## 3. Longueur des questions et réponses

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['q_len'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(df['q_len'].median(), color='red', linestyle='--', label=f"Médiane : {df['q_len'].median():.0f}")
axes[0].set_title('Longueur des questions (mots)', fontweight='bold')
axes[0].set_xlabel('Nombre de mots')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

axes[1].hist(df['a_len'], bins=40, color='darkorange', edgecolor='white')
axes[1].axvline(df['a_len'].median(), color='red', linestyle='--', label=f"Médiane : {df['a_len'].median():.0f}")
axes[1].set_title('Longueur des réponses (mots)', fontweight='bold')
axes[1].set_xlabel('Nombre de mots')
axes[1].legend()

plt.suptitle('Distribution des longueurs de texte', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'nlp', 'preprocessing', 'results', 'text_lengths.png'), dpi=150)
plt.show()

## 4. Split Train / Val / Test par spécialité

In [ ]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Category'])
val, test   = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['Category'])

split_df = pd.DataFrame({
    'Train': train['Category'].value_counts(),
    'Val':   val['Category'].value_counts(),
    'Test':  test['Category'].value_counts()
}).fillna(0).astype(int)

split_df.sort_values('Train', ascending=True).plot(
    kind='barh', stacked=True, figsize=(14, 8),
    color=['#2196F3', '#FF9800', '#4CAF50']
)
plt.title('Répartition Train / Val / Test par spécialité', fontsize=13, fontweight='bold')
plt.xlabel('Nombre de questions')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'nlp', 'preprocessing', 'results', 'train_val_test_split.png'), dpi=150)
plt.show()

print(f"Train : {len(train):,}  |  Val : {len(val):,}  |  Test : {len(test):,}")

## 5. Statistiques résumées

In [ ]:
print("=== Statistiques générales ===")
print(f"Total lignes       : {len(df):,}")
print(f"Spécialités        : {df['Category'].nunique()}")
print(f"Moy. mots/question : {df['q_len'].mean():.1f}")
print(f"Moy. mots/réponse  : {df['a_len'].mean():.1f}")
print(f"Classe dominante   : {counts.index[0]} ({counts.iloc[0]} ex.)")
print(f"Classe minoritaire : {counts.index[-1]} ({counts.iloc[-1]} ex.)")
print(f"\nRépartition :")
print(f"  Train : {len(train):,} ({len(train)/len(df)*100:.1f}%)")
print(f"  Val   : {len(val):,} ({len(val)/len(df)*100:.1f}%)")
print(f"  Test  : {len(test):,} ({len(test)/len(df)*100:.1f}%)")